In [ ]:
import os
os.chdir('../..')

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from loguru import logger
import kmedoids 
import chemiscope
import persim

from src.datasets import QM9Dataset
from src.features import get_features_xyz, get_raw_xyz_features
from src.non_euclidean import Grassmann, Riemann, PersistentHomology, Wasserstein
from src.helper_functions import align_frames_to_dist_matrix, get_distances

In [ ]:
qm9 = QM9Dataset()
qm9.load()
frames = qm9.export_subset_xyz()

2026-03-05 16:59:09.606 | INFO     | src.datasets:load:71 - Loading QM9 from data/QM9/dataset_cleaned.csv...
2026-03-05 16:59:10.631 | DEBUG    | src.datasets:export_subset_xyz:483 - Skipping qm9_686: Embedding failed.
2026-03-05 16:59:13.215 | SUCCESS  | src.datasets:export_subset_xyz:491 - Saved 1999 molecules to data/QM9/qm9_subset.xyz (failed: 1, requested: 2000).


In [ ]:
distance_matrices = get_distances()

2026-03-05 16:59:18.139 | INFO     | __main__:<module>:22 - Computing distance matrices (this may take a while)...
2026-03-05 16:59:18.140 | INFO     | src.non_euclidean:distance_matrix:215 - Computing Grassmann distance matrix for 1999 frames (k=3, method='qr').
Grassmann distances: 100%|██████████| 1997001/1997001 [02:28<00:00, 13488.51pair/s]
2026-03-05 17:01:46.244 | DEBUG    | src.non_euclidean:distance_matrix:230 - Finished Grassmann distance matrix computation.
2026-03-05 17:01:46.245 | INFO     | src.non_euclidean:distance_matrix:262 - Computing Riemannian distance matrix for 1999 frames (metric_type='log-euclidean').
Riemannian distances:   6%|▌         | 123295/1997001 [00:03<00:47, 39847.57pair/s]


KeyboardInterrupt: 

In [ ]:
def evaluate_k_for_distance_matrices(
    distance_sets,
    k_range=range(2, 11),
    random_state=42,
    max_silhouette_samples=10_000,
):
    """
    Evaluate candidate k values for one or more precomputed distance matrices.

    Parameters
    ----------
    distance_sets : dict[str, np.ndarray]
        Mapping from name -> square pairwise distance matrix.
    k_range : iterable[int]
        Cluster counts to evaluate.
    random_state : int
        Seed for reproducibility.
    max_silhouette_samples : int
        Max number of samples used for silhouette score.
    """
    results = {name: {'cost': [], 'silhouette': []} for name in distance_sets}

    for name, dist_matrix in distance_sets.items():
        dist_matrix = np.asarray(dist_matrix)
        if dist_matrix.ndim != 2 or dist_matrix.shape[0] != dist_matrix.shape[1]:
            raise ValueError(f"Distance matrix '{name}' must be square. Got {dist_matrix.shape}.")

        n = dist_matrix.shape[0]
        valid_k = [k for k in k_range if 2 <= k < n]
        if not valid_k:
            raise ValueError(
                f"No valid k values for '{name}'. Need 2 <= k < n_samples (n={n})."
            )

        print(f"Processing {name} (n={n})...")
        for k in valid_k:
            model = kmedoids.KMedoids(n_clusters=k, random_state=random_state)
            labels = model.fit_predict(dist_matrix)

            # K-Medoids objective. Prefer library value; fallback to direct medoid cost.
            if hasattr(model, 'inertia_') and model.inertia_ is not None:
                cost = float(model.inertia_)
            elif hasattr(model, 'medoid_indices_') and model.medoid_indices_ is not None:
                medoid_indices = np.asarray(model.medoid_indices_, dtype=int)
                cost = float(np.sum(np.min(dist_matrix[:, medoid_indices], axis=1)))
            else:
                cost = float('nan')
            results[name]['cost'].append(cost)

            if n > max_silhouette_samples:
                sil = silhouette_score(
                    dist_matrix,
                    labels,
                    metric='precomputed',
                    sample_size=max_silhouette_samples,
                    random_state=random_state,
                )
            else:
                sil = silhouette_score(dist_matrix, labels, metric='precomputed')
            results[name]['silhouette'].append(float(sil))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

    for name, metrics in results.items():
        n = np.asarray(distance_sets[name]).shape[0]
        valid_k = [k for k in k_range if 2 <= k < n]
        ax1.plot(valid_k, metrics['cost'], marker='o', label=name)

    ax1.set_title('K-Medoids Objective (Cost)')
    ax1.set_xlabel('Number of Clusters (k)')
    ax1.set_ylabel('Cost (Lower is better)')
    ax1.legend()
    ax1.grid(True)

    for name, metrics in results.items():
        n = np.asarray(distance_sets[name]).shape[0]
        valid_k = [k for k in k_range if 2 <= k < n]
        ax2.plot(valid_k, metrics['silhouette'], marker='s', linestyle='--', label=name)

    ax2.set_title('Silhouette Score (Precomputed Distances)')
    ax2.set_xlabel('Number of Clusters (k)')
    ax2.set_ylabel('Silhouette Score (Higher is better)')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

    best_k = {
        name: [k for k in k_range if 2 <= k < np.asarray(distance_sets[name]).shape[0]][
            int(np.argmax(metrics['silhouette']))
        ]
        for name, metrics in results.items()
    }

    summary = pl.DataFrame(
        {
            'name': list(best_k.keys()),
            'best_k_by_silhouette': list(best_k.values()),
            'best_silhouette': [max(results[name]['silhouette']) for name in best_k],
        }
    )

    display(summary)
    return results, summary


In [ ]:
def make_clustering(frames, rotational_invariant=False, dist_matrix=None, metric_name="euclidean"):
    """
    Unified clustering function handling both feature matrices and precomputed distance matrices.
    """
    active_frames = align_frames_to_dist_matrix(frames, dist_matrix=dist_matrix)
    true_labels = [f.info.get('mol_id', '') for f in active_frames]
    smiles_list = [f.info.get('smiles', '') for f in active_frames]
    base_or_pertubated = [f.info.get('frame_type', 'base') for f in active_frames]

    n_clusters = 5

    if dist_matrix is not None:
        print(f"Using precomputed {metric_name} distance matrix (Shape: {dist_matrix.shape})")
        
        clustering = kmedoids.KMedoids(n_clusters)
        cluster_labels = clustering.fit_predict(dist_matrix)
        
        tsne = TSNE(
            n_components=2, 
            metric='precomputed', 
            init='random', 
            random_state=42, 
            perplexity=30
        )
        X_tsne = tsne.fit_transform(dist_matrix)
        
        suffix = metric_name

    else:
        print("Using feature-based representation...")
        if rotational_invariant:
            X = get_features_xyz(active_frames)
            suffix = "invariant"
        else:
            X = get_raw_xyz_features(active_frames)
            suffix = "raw"
            
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(X)

        tsne = TSNE(
            n_components=2, 
            random_state=42, 
            perplexity=30, 
            init='pca'
        )
        X_tsne = tsne.fit_transform(X)

    ari_score = adjusted_rand_score(true_labels, cluster_labels)
    print(f"\nClustering Performance (Adjusted Rand Index) [{suffix}]: {ari_score:.4f}")

    analysis_df = pl.DataFrame({
        "cluster": cluster_labels,
        "true_mol_id": true_labels,
        "smiles": smiles_list
    })

    summary_table = (
        analysis_df.group_by("cluster")
        .agg([
            pl.col("true_mol_id").n_unique().alias("unique_mols"),
            pl.col("true_mol_id").unique().alias("mol_ids_in_cluster"),
            pl.col("smiles").first().alias("representative_smiles"),
            pl.len().alias("total_samples")
        ])
        .sort("cluster")
    )

    pl.Config.set_fmt_str_lengths(1000)
    pl.Config.set_tbl_rows(100)
    pl.Config.set_fmt_table_cell_list_len(50)
    print("\nCluster Composition Summary:")
    display(summary_table)

    structures = active_frames

    properties = {
        "t-SNE 1": X_tsne[:, 0],
        "t-SNE 2": X_tsne[:, 1],
        "Cluster": cluster_labels,
        "True Mol ID": true_labels,
        "SMILES": smiles_list,
        "Frame Type": base_or_pertubated
    }

    output_path = f"report/qm9/figures/grassmann/chemiscope_clustering_{suffix}.json.gz"

    chemiscope.write_input(
        output_path,
        properties=properties,
        structures=structures,
    )
    
    print(f"\nChemiscope file successfully saved to: {output_path}")

    return summary_table, output_path

In [ ]:
summary_table, output_path = make_clustering(frames, dist_matrix = dist_matrix_grassmann)

Using precomputed euclidean distance matrix (Shape: (1999, 1999))

Clustering Performance (Adjusted Rand Index) [euclidean]: 0.0000

Cluster Composition Summary:


cluster,unique_mols,mol_ids_in_cluster,representative_smiles,total_samples
u64,u32,list[str],str,u32
0,408,"[""qm9_1"", ""qm9_2"", ""qm9_3"", ""qm9_4"", ""qm9_5"", ""qm9_7"", ""qm9_9"", ""qm9_24"", ""qm9_25"", ""qm9_27"", ""qm9_77"", ""qm9_79"", ""qm9_81"", ""qm9_132"", ""qm9_167"", ""qm9_169"", ""qm9_170"", ""qm9_171"", ""qm9_172"", ""qm9_173"", ""qm9_180"", ""qm9_223"", ""qm9_243"", ""qm9_249"", ""qm9_251"", ""qm9_253"", ""qm9_256"", ""qm9_264"", ""qm9_265"", ""qm9_269"", ""qm9_270"", ""qm9_275"", ""qm9_276"", ""qm9_279"", ""qm9_281"", ""qm9_285"", ""qm9_287"", ""qm9_288"", ""qm9_290"", ""qm9_292"", ""qm9_302"", ""qm9_303"", ""qm9_304"", ""qm9_348"", ""qm9_350"", ""qm9_360"", ""qm9_362"", ""qm9_380"", ""qm9_382"", … ""qm9_2235""]","""[H]N([H])[H]""",408
1,447,"[""qm9_17"", ""qm9_21"", ""qm9_30"", ""qm9_46"", ""qm9_48"", ""qm9_65"", ""qm9_67"", ""qm9_69"", ""qm9_78"", ""qm9_80"", ""qm9_84"", ""qm9_91"", ""qm9_97"", ""qm9_115"", ""qm9_116"", ""qm9_119"", ""qm9_124"", ""qm9_130"", ""qm9_131"", ""qm9_146"", ""qm9_149"", ""qm9_150"", ""qm9_158"", ""qm9_160"", ""qm9_175"", ""qm9_181"", ""qm9_185"", ""qm9_187"", ""qm9_188"", ""qm9_195"", ""qm9_196"", ""qm9_198"", ""qm9_199"", ""qm9_217"", ""qm9_221"", ""qm9_225"", ""qm9_226"", ""qm9_227"", ""qm9_230"", ""qm9_232"", ""qm9_233"", ""qm9_236"", ""qm9_238"", ""qm9_244"", ""qm9_257"", ""qm9_259"", ""qm9_261"", ""qm9_263"", ""qm9_272"", … ""qm9_2232""]","""[H]C([H])([H])C(=O)C([H])([H])[H]""",447
2,373,"[""qm9_20"", ""qm9_39"", ""qm9_40"", ""qm9_42"", ""qm9_44"", ""qm9_61"", ""qm9_70"", ""qm9_73"", ""qm9_82"", ""qm9_83"", ""qm9_92"", ""qm9_93"", ""qm9_94"", ""qm9_96"", ""qm9_98"", ""qm9_99"", ""qm9_100"", ""qm9_101"", ""qm9_114"", ""qm9_122"", ""qm9_125"", ""qm9_126"", ""qm9_127"", ""qm9_128"", ""qm9_145"", ""qm9_147"", ""qm9_148"", ""qm9_151"", ""qm9_152"", ""qm9_157"", ""qm9_177"", ""qm9_178"", ""qm9_179"", ""qm9_184"", ""qm9_186"", ""qm9_197"", ""qm9_215"", ""qm9_216"", ""qm9_228"", ""qm9_240"", ""qm9_246"", ""qm9_247"", ""qm9_255"", ""qm9_262"", ""qm9_277"", ""qm9_280"", ""qm9_295"", ""qm9_335"", ""qm9_338"", … ""qm9_2215""]","""[H]C([H])([H])C([H])(C([H])([H])[H])C([H])([H])[H]""",373
3,464,"[""qm9_11"", ""qm9_38"", ""qm9_53"", ""qm9_54"", ""qm9_118"", ""qm9_129"", ""qm9_134"", ""qm9_155"", ""qm9_176"", ""qm9_213"", ""qm9_220"", ""qm9_222"", ""qm9_224"", ""qm9_248"", ""qm9_252"", ""qm9_254"", ""qm9_266"", ""qm9_267"", ""qm9_268"", ""qm9_282"", ""qm9_283"", ""qm9_284"", ""qm9_286"", ""qm9_291"", ""qm9_294"", ""qm9_296"", ""qm9_299"", ""qm9_301"", ""qm9_306"", ""qm9_375"", ""qm9_383"", ""qm9_401"", ""qm9_402"", ""qm9_404"", ""qm9_405"", ""qm9_406"", ""qm9_407"", ""qm9_409"", ""qm9_423"", ""qm9_452"", ""qm9_456"", ""qm9_463"", ""qm9_464"", ""qm9_465"", ""qm9_510"", ""qm9_512"", ""qm9_514"", ""qm9_516"", ""qm9_528"", … ""qm9_2233""]","""[H]C(=O)N([H])[H]""",464
4,307,"[""qm9_0"", ""qm9_6"", ""qm9_8"", ""qm9_10"", ""qm9_12"", ""qm9_13"", ""qm9_14"", ""qm9_15"", ""qm9_16"", ""qm9_18"", ""qm9_19"", ""qm9_23"", ""qm9_26"", ""qm9_29"", ""qm9_31"", ""qm9_32"", ""qm9_33"", ""qm9_34"", ""qm9_35"", ""qm9_36"", ""qm9_37"", ""qm9_41"", ""qm9_43"", ""qm9_45"", ""qm9_47"", ""qm9_49"", ""qm9_50"", ""qm9_51"", ""qm9_52"", ""qm9_55"", ""qm9_56"", ""qm9_57"", ""qm9_58"", ""qm9_59"", ""qm9_62"", ""qm9_63"", ""qm9_64"", ""qm9_66"", ""qm9_68"", ""qm9_71"", ""qm9_72"", ""qm9_74"", ""qm9_75"", ""qm9_76"", ""qm9_85"", ""qm9_86"", ""qm9_87"", ""qm9_88"", ""qm9_89"", … ""qm9_2206""]","""[H]C([H])([H])[H]""",307



Chemiscope file successfully saved to: report/qm9/figures/grassmann/chemiscope_clustering_euclidean.json.gz


In [ ]:
chemiscope.show_input(output_path)

<ChemiscopeWidget(meta={'name': 'chemiscope_clustering_euclidean'}, structures=[{'size': 5, 'data': 'structure…

# Riemann

In [ ]:
summary_table, output_path = make_clustering(frames, dist_matrix = dist_matrix_grassmann)
chemiscope.show_input(output_path)

Using precomputed euclidean distance matrix (Shape: (1999, 1999))

Clustering Performance (Adjusted Rand Index) [euclidean]: 0.0000

Cluster Composition Summary:


cluster,unique_mols,mol_ids_in_cluster,representative_smiles,total_samples
u64,u32,list[str],str,u32
0,464,"[""qm9_11"", ""qm9_38"", ""qm9_53"", ""qm9_54"", ""qm9_118"", ""qm9_129"", ""qm9_134"", ""qm9_155"", ""qm9_176"", ""qm9_213"", ""qm9_220"", ""qm9_222"", ""qm9_224"", ""qm9_248"", ""qm9_252"", ""qm9_254"", ""qm9_266"", ""qm9_267"", ""qm9_268"", ""qm9_282"", ""qm9_283"", ""qm9_284"", ""qm9_286"", ""qm9_291"", ""qm9_294"", ""qm9_296"", ""qm9_299"", ""qm9_301"", ""qm9_306"", ""qm9_375"", ""qm9_383"", ""qm9_401"", ""qm9_402"", ""qm9_404"", ""qm9_405"", ""qm9_406"", ""qm9_407"", ""qm9_409"", ""qm9_423"", ""qm9_452"", ""qm9_456"", ""qm9_463"", ""qm9_464"", ""qm9_465"", ""qm9_510"", ""qm9_512"", ""qm9_514"", ""qm9_516"", ""qm9_528"", … ""qm9_2233""]","""[H]C(=O)N([H])[H]""",464
1,408,"[""qm9_1"", ""qm9_2"", ""qm9_3"", ""qm9_4"", ""qm9_5"", ""qm9_7"", ""qm9_9"", ""qm9_24"", ""qm9_25"", ""qm9_27"", ""qm9_77"", ""qm9_79"", ""qm9_81"", ""qm9_132"", ""qm9_167"", ""qm9_169"", ""qm9_170"", ""qm9_171"", ""qm9_172"", ""qm9_173"", ""qm9_180"", ""qm9_223"", ""qm9_243"", ""qm9_249"", ""qm9_251"", ""qm9_253"", ""qm9_256"", ""qm9_264"", ""qm9_265"", ""qm9_269"", ""qm9_270"", ""qm9_275"", ""qm9_276"", ""qm9_279"", ""qm9_281"", ""qm9_285"", ""qm9_287"", ""qm9_288"", ""qm9_290"", ""qm9_292"", ""qm9_302"", ""qm9_303"", ""qm9_304"", ""qm9_348"", ""qm9_350"", ""qm9_360"", ""qm9_362"", ""qm9_380"", ""qm9_382"", … ""qm9_2235""]","""[H]N([H])[H]""",408
2,307,"[""qm9_0"", ""qm9_6"", ""qm9_8"", ""qm9_10"", ""qm9_12"", ""qm9_13"", ""qm9_14"", ""qm9_15"", ""qm9_16"", ""qm9_18"", ""qm9_19"", ""qm9_23"", ""qm9_26"", ""qm9_29"", ""qm9_31"", ""qm9_32"", ""qm9_33"", ""qm9_34"", ""qm9_35"", ""qm9_36"", ""qm9_37"", ""qm9_41"", ""qm9_43"", ""qm9_45"", ""qm9_47"", ""qm9_49"", ""qm9_50"", ""qm9_51"", ""qm9_52"", ""qm9_55"", ""qm9_56"", ""qm9_57"", ""qm9_58"", ""qm9_59"", ""qm9_62"", ""qm9_63"", ""qm9_64"", ""qm9_66"", ""qm9_68"", ""qm9_71"", ""qm9_72"", ""qm9_74"", ""qm9_75"", ""qm9_76"", ""qm9_85"", ""qm9_86"", ""qm9_87"", ""qm9_88"", ""qm9_89"", … ""qm9_2206""]","""[H]C([H])([H])[H]""",307
3,373,"[""qm9_20"", ""qm9_39"", ""qm9_40"", ""qm9_42"", ""qm9_44"", ""qm9_61"", ""qm9_70"", ""qm9_73"", ""qm9_82"", ""qm9_83"", ""qm9_92"", ""qm9_93"", ""qm9_94"", ""qm9_96"", ""qm9_98"", ""qm9_99"", ""qm9_100"", ""qm9_101"", ""qm9_114"", ""qm9_122"", ""qm9_125"", ""qm9_126"", ""qm9_127"", ""qm9_128"", ""qm9_145"", ""qm9_147"", ""qm9_148"", ""qm9_151"", ""qm9_152"", ""qm9_157"", ""qm9_177"", ""qm9_178"", ""qm9_179"", ""qm9_184"", ""qm9_186"", ""qm9_197"", ""qm9_215"", ""qm9_216"", ""qm9_228"", ""qm9_240"", ""qm9_246"", ""qm9_247"", ""qm9_255"", ""qm9_262"", ""qm9_277"", ""qm9_280"", ""qm9_295"", ""qm9_335"", ""qm9_338"", … ""qm9_2215""]","""[H]C([H])([H])C([H])(C([H])([H])[H])C([H])([H])[H]""",373
4,447,"[""qm9_17"", ""qm9_21"", ""qm9_30"", ""qm9_46"", ""qm9_48"", ""qm9_65"", ""qm9_67"", ""qm9_69"", ""qm9_78"", ""qm9_80"", ""qm9_84"", ""qm9_91"", ""qm9_97"", ""qm9_115"", ""qm9_116"", ""qm9_119"", ""qm9_124"", ""qm9_130"", ""qm9_131"", ""qm9_146"", ""qm9_149"", ""qm9_150"", ""qm9_158"", ""qm9_160"", ""qm9_175"", ""qm9_181"", ""qm9_185"", ""qm9_187"", ""qm9_188"", ""qm9_195"", ""qm9_196"", ""qm9_198"", ""qm9_199"", ""qm9_217"", ""qm9_221"", ""qm9_225"", ""qm9_226"", ""qm9_227"", ""qm9_230"", ""qm9_232"", ""qm9_233"", ""qm9_236"", ""qm9_238"", ""qm9_244"", ""qm9_257"", ""qm9_259"", ""qm9_261"", ""qm9_263"", ""qm9_272"", … ""qm9_2232""]","""[H]C([H])([H])C(=O)C([H])([H])[H]""",447



Chemiscope file successfully saved to: report/qm9/figures/grassmann/chemiscope_clustering_euclidean.json.gz


<ChemiscopeWidget(meta={'name': 'chemiscope_clustering_euclidean'}, structures=[{'size': 5, 'data': 'structure…

In [ ]:
summary_table, output_path = make_clustering(frames, dist_matrix = dist_matrix_grassmann)
chemiscope.show_input(output_path)

Using precomputed euclidean distance matrix (Shape: (1999, 1999))

Clustering Performance (Adjusted Rand Index) [euclidean]: 0.0000

Cluster Composition Summary:


cluster,unique_mols,mol_ids_in_cluster,representative_smiles,total_samples
u64,u32,list[str],str,u32
0,379,"[""qm9_1"", ""qm9_2"", ""qm9_3"", ""qm9_4"", ""qm9_5"", ""qm9_20"", ""qm9_24"", ""qm9_26"", ""qm9_56"", ""qm9_82"", ""qm9_91"", ""qm9_130"", ""qm9_145"", ""qm9_146"", ""qm9_155"", ""qm9_165"", ""qm9_167"", ""qm9_168"", ""qm9_169"", ""qm9_170"", ""qm9_171"", ""qm9_172"", ""qm9_173"", ""qm9_177"", ""qm9_180"", ""qm9_247"", ""qm9_256"", ""qm9_261"", ""qm9_269"", ""qm9_275"", ""qm9_276"", ""qm9_302"", ""qm9_312"", ""qm9_313"", ""qm9_319"", ""qm9_348"", ""qm9_359"", ""qm9_380"", ""qm9_382"", ""qm9_385"", ""qm9_409"", ""qm9_424"", ""qm9_449"", ""qm9_453"", ""qm9_468"", ""qm9_469"", ""qm9_476"", ""qm9_535"", ""qm9_538"", … ""qm9_2235""]","""[H]N([H])[H]""",379
1,491,"[""qm9_7"", ""qm9_9"", ""qm9_11"", ""qm9_25"", ""qm9_27"", ""qm9_53"", ""qm9_93"", ""qm9_118"", ""qm9_129"", ""qm9_134"", ""qm9_176"", ""qm9_213"", ""qm9_220"", ""qm9_221"", ""qm9_222"", ""qm9_224"", ""qm9_248"", ""qm9_249"", ""qm9_252"", ""qm9_254"", ""qm9_264"", ""qm9_265"", ""qm9_266"", ""qm9_267"", ""qm9_281"", ""qm9_282"", ""qm9_283"", ""qm9_284"", ""qm9_286"", ""qm9_291"", ""qm9_294"", ""qm9_301"", ""qm9_303"", ""qm9_304"", ""qm9_306"", ""qm9_350"", ""qm9_374"", ""qm9_375"", ""qm9_383"", ""qm9_401"", ""qm9_402"", ""qm9_404"", ""qm9_405"", ""qm9_406"", ""qm9_407"", ""qm9_422"", ""qm9_452"", ""qm9_454"", ""qm9_456"", … ""qm9_2233""]","""[H]OC([H])([H])[H]""",491
2,404,"[""qm9_21"", ""qm9_39"", ""qm9_40"", ""qm9_44"", ""qm9_61"", ""qm9_63"", ""qm9_65"", ""qm9_67"", ""qm9_68"", ""qm9_69"", ""qm9_70"", ""qm9_73"", ""qm9_83"", ""qm9_94"", ""qm9_97"", ""qm9_99"", ""qm9_114"", ""qm9_116"", ""qm9_117"", ""qm9_126"", ""qm9_127"", ""qm9_128"", ""qm9_147"", ""qm9_148"", ""qm9_150"", ""qm9_179"", ""qm9_184"", ""qm9_186"", ""qm9_195"", ""qm9_197"", ""qm9_198"", ""qm9_215"", ""qm9_217"", ""qm9_226"", ""qm9_228"", ""qm9_233"", ""qm9_240"", ""qm9_246"", ""qm9_259"", ""qm9_272"", ""qm9_277"", ""qm9_330"", ""qm9_338"", ""qm9_343"", ""qm9_344"", ""qm9_345"", ""qm9_353"", ""qm9_355"", ""qm9_357"", … ""qm9_2224""]","""[H]OC([H])(C([H])([H])[H])C([H])([H])[H]""",404
3,423,"[""qm9_0"", ""qm9_6"", ""qm9_8"", ""qm9_10"", ""qm9_12"", ""qm9_13"", ""qm9_14"", ""qm9_15"", ""qm9_16"", ""qm9_17"", ""qm9_18"", ""qm9_19"", ""qm9_23"", ""qm9_29"", ""qm9_30"", ""qm9_31"", ""qm9_32"", ""qm9_33"", ""qm9_34"", ""qm9_35"", ""qm9_36"", ""qm9_37"", ""qm9_41"", ""qm9_42"", ""qm9_43"", ""qm9_45"", ""qm9_47"", ""qm9_48"", ""qm9_49"", ""qm9_50"", ""qm9_51"", ""qm9_52"", ""qm9_55"", ""qm9_57"", ""qm9_58"", ""qm9_59"", ""qm9_62"", ""qm9_64"", ""qm9_66"", ""qm9_71"", ""qm9_72"", ""qm9_74"", ""qm9_75"", ""qm9_76"", ""qm9_85"", ""qm9_86"", ""qm9_87"", ""qm9_88"", ""qm9_89"", … ""qm9_2232""]","""[H]C([H])([H])[H]""",423
4,302,"[""qm9_38"", ""qm9_46"", ""qm9_54"", ""qm9_77"", ""qm9_78"", ""qm9_79"", ""qm9_80"", ""qm9_81"", ""qm9_84"", ""qm9_92"", ""qm9_96"", ""qm9_98"", ""qm9_101"", ""qm9_119"", ""qm9_124"", ""qm9_131"", ""qm9_132"", ""qm9_149"", ""qm9_158"", ""qm9_164"", ""qm9_175"", ""qm9_178"", ""qm9_216"", ""qm9_223"", ""qm9_225"", ""qm9_227"", ""qm9_230"", ""qm9_236"", ""qm9_243"", ""qm9_251"", ""qm9_253"", ""qm9_255"", ""qm9_257"", ""qm9_262"", ""qm9_268"", ""qm9_270"", ""qm9_279"", ""qm9_280"", ""qm9_285"", ""qm9_287"", ""qm9_288"", ""qm9_290"", ""qm9_292"", ""qm9_293"", ""qm9_295"", ""qm9_296"", ""qm9_299"", ""qm9_305"", ""qm9_309"", … ""qm9_2230""]","""[H]C([H])([H])C([H])([H])C([H])([H])C([H])([H])[H]""",302



Chemiscope file successfully saved to: report/qm9/figures/grassmann/chemiscope_clustering_euclidean.json.gz


<ChemiscopeWidget(meta={'name': 'chemiscope_clustering_euclidean'}, structures=[{'size': 5, 'data': 'structure…

# Earths movers distance

In [ ]:
summary_table, output_path = make_clustering(frames, dist_matrix = dist_matrix_grassmann)
chemiscope.show_input(output_path)

Using precomputed euclidean distance matrix (Shape: (1999, 1999))

Clustering Performance (Adjusted Rand Index) [euclidean]: 0.0000

Cluster Composition Summary:


cluster,unique_mols,mol_ids_in_cluster,representative_smiles,total_samples
u64,u32,list[str],str,u32
0,307,"[""qm9_0"", ""qm9_6"", ""qm9_8"", ""qm9_10"", ""qm9_12"", ""qm9_13"", ""qm9_14"", ""qm9_15"", ""qm9_16"", ""qm9_18"", ""qm9_19"", ""qm9_23"", ""qm9_26"", ""qm9_29"", ""qm9_31"", ""qm9_32"", ""qm9_33"", ""qm9_34"", ""qm9_35"", ""qm9_36"", ""qm9_37"", ""qm9_41"", ""qm9_43"", ""qm9_45"", ""qm9_47"", ""qm9_49"", ""qm9_50"", ""qm9_51"", ""qm9_52"", ""qm9_55"", ""qm9_56"", ""qm9_57"", ""qm9_58"", ""qm9_59"", ""qm9_62"", ""qm9_63"", ""qm9_64"", ""qm9_66"", ""qm9_68"", ""qm9_71"", ""qm9_72"", ""qm9_74"", ""qm9_75"", ""qm9_76"", ""qm9_85"", ""qm9_86"", ""qm9_87"", ""qm9_88"", ""qm9_89"", … ""qm9_2206""]","""[H]C([H])([H])[H]""",307
1,373,"[""qm9_20"", ""qm9_39"", ""qm9_40"", ""qm9_42"", ""qm9_44"", ""qm9_61"", ""qm9_70"", ""qm9_73"", ""qm9_82"", ""qm9_83"", ""qm9_92"", ""qm9_93"", ""qm9_94"", ""qm9_96"", ""qm9_98"", ""qm9_99"", ""qm9_100"", ""qm9_101"", ""qm9_114"", ""qm9_122"", ""qm9_125"", ""qm9_126"", ""qm9_127"", ""qm9_128"", ""qm9_145"", ""qm9_147"", ""qm9_148"", ""qm9_151"", ""qm9_152"", ""qm9_157"", ""qm9_177"", ""qm9_178"", ""qm9_179"", ""qm9_184"", ""qm9_186"", ""qm9_197"", ""qm9_215"", ""qm9_216"", ""qm9_228"", ""qm9_240"", ""qm9_246"", ""qm9_247"", ""qm9_255"", ""qm9_262"", ""qm9_277"", ""qm9_280"", ""qm9_295"", ""qm9_335"", ""qm9_338"", … ""qm9_2215""]","""[H]C([H])([H])C([H])(C([H])([H])[H])C([H])([H])[H]""",373
2,464,"[""qm9_11"", ""qm9_38"", ""qm9_53"", ""qm9_54"", ""qm9_118"", ""qm9_129"", ""qm9_134"", ""qm9_155"", ""qm9_176"", ""qm9_213"", ""qm9_220"", ""qm9_222"", ""qm9_224"", ""qm9_248"", ""qm9_252"", ""qm9_254"", ""qm9_266"", ""qm9_267"", ""qm9_268"", ""qm9_282"", ""qm9_283"", ""qm9_284"", ""qm9_286"", ""qm9_291"", ""qm9_294"", ""qm9_296"", ""qm9_299"", ""qm9_301"", ""qm9_306"", ""qm9_375"", ""qm9_383"", ""qm9_401"", ""qm9_402"", ""qm9_404"", ""qm9_405"", ""qm9_406"", ""qm9_407"", ""qm9_409"", ""qm9_423"", ""qm9_452"", ""qm9_456"", ""qm9_463"", ""qm9_464"", ""qm9_465"", ""qm9_510"", ""qm9_512"", ""qm9_514"", ""qm9_516"", ""qm9_528"", … ""qm9_2233""]","""[H]C(=O)N([H])[H]""",464
3,408,"[""qm9_1"", ""qm9_2"", ""qm9_3"", ""qm9_4"", ""qm9_5"", ""qm9_7"", ""qm9_9"", ""qm9_24"", ""qm9_25"", ""qm9_27"", ""qm9_77"", ""qm9_79"", ""qm9_81"", ""qm9_132"", ""qm9_167"", ""qm9_169"", ""qm9_170"", ""qm9_171"", ""qm9_172"", ""qm9_173"", ""qm9_180"", ""qm9_223"", ""qm9_243"", ""qm9_249"", ""qm9_251"", ""qm9_253"", ""qm9_256"", ""qm9_264"", ""qm9_265"", ""qm9_269"", ""qm9_270"", ""qm9_275"", ""qm9_276"", ""qm9_279"", ""qm9_281"", ""qm9_285"", ""qm9_287"", ""qm9_288"", ""qm9_290"", ""qm9_292"", ""qm9_302"", ""qm9_303"", ""qm9_304"", ""qm9_348"", ""qm9_350"", ""qm9_360"", ""qm9_362"", ""qm9_380"", ""qm9_382"", … ""qm9_2235""]","""[H]N([H])[H]""",408
4,447,"[""qm9_17"", ""qm9_21"", ""qm9_30"", ""qm9_46"", ""qm9_48"", ""qm9_65"", ""qm9_67"", ""qm9_69"", ""qm9_78"", ""qm9_80"", ""qm9_84"", ""qm9_91"", ""qm9_97"", ""qm9_115"", ""qm9_116"", ""qm9_119"", ""qm9_124"", ""qm9_130"", ""qm9_131"", ""qm9_146"", ""qm9_149"", ""qm9_150"", ""qm9_158"", ""qm9_160"", ""qm9_175"", ""qm9_181"", ""qm9_185"", ""qm9_187"", ""qm9_188"", ""qm9_195"", ""qm9_196"", ""qm9_198"", ""qm9_199"", ""qm9_217"", ""qm9_221"", ""qm9_225"", ""qm9_226"", ""qm9_227"", ""qm9_230"", ""qm9_232"", ""qm9_233"", ""qm9_236"", ""qm9_238"", ""qm9_244"", ""qm9_257"", ""qm9_259"", ""qm9_261"", ""qm9_263"", ""qm9_272"", … ""qm9_2232""]","""[H]C([H])([H])C(=O)C([H])([H])[H]""",447



Chemiscope file successfully saved to: report/qm9/figures/grassmann/chemiscope_clustering_euclidean.json.gz


<ChemiscopeWidget(meta={'name': 'chemiscope_clustering_euclidean'}, structures=[{'size': 5, 'data': 'structure…

# Persistent homology

In [ ]:
persistent_dist_matrix = PersistentHomology.distance_matrix(frames, metric="bottleneck")

2026-03-05 15:58:23.881 | INFO     | src.non_euclidean:distance_matrix:136 - Computing persistent homology distance matrix for 1999 frames (backend='ripser', metric='bottleneck', max_homology_dim=2, dims=(0, 1, 2)).
2026-03-05 15:58:23.881 | INFO     | src.non_euclidean:compute_persistence_diagrams:82 - Computing persistence diagrams for 1999 frames (backend='ripser', max_homology_dim=2).
/Users/karlfindhansen/Desktop/DTU/Kandidat/Thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/.venv/lib/python3.12/site-packages/ripser/ripser.py:251: UserWarning: The input matrix is square, but the distance_matrix flag is off.  Did you mean to indicate that this was a distance matrix?
  warnings.warn(
/Users/karlfindhansen/Desktop/DTU/Kandidat/Thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/.venv/lib/python3.12/site-packages/ripser/ripser.py:251: UserWarning: The input matrix is square, but the distance_matrix flag is off.  Did you mean to indicate that this was a distance mat

KeyboardInterrupt: 

In [ ]:
summary_persistent_homology, path_persistent_homology = make_clustering(
    frames, 
    dist_matrix=persistent_dist_matrix,
)

chemiscope.show_input(path_persistent_homology)